<a href="https://colab.research.google.com/github/hyunkyung31/DeepLearningProject/blob/main/0715_InceptionV3/yuri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import urllib.request
import zipfile
dataset_url = "/content/drive/MyDrive/바이오메디컬AI/🐥new_2_project/04_DL_project/02_dataset/CADICA.zip"
with zipfile.ZipFile("/content/drive/MyDrive/바이오메디컬AI/🐥new_2_project/04_DL_project/02_dataset/CADICA.zip", "r") as zip_ref:
     zip_ref.extractall("./CADICA")
     print("압축 해제 완료!")

압축 해제 완료!


In [3]:
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader
csv_path = "/content/drive/MyDrive/바이오메디컬AI/🐥new_2_project/04_DL_project/08_전처리/common_split.csv"
df = pd.read_csv(csv_path)

base = Path("/content/CADICA/CADICA/selectedVideos")

def expand_to_frames(row):
    v_dir = base / row.patient_id / row.video_id / "input"
    if not v_dir.exists():
        return []
    frame_rows = []
    for png_path in sorted(v_dir.glob("*.png")):
        frame_rows.append({
            "patient_id": row.patient_id,
            "video_id": row.video_id,
            "frame_path": str(png_path),
            "frame_id": png_path.stem,          # 예: p1_v2_00015
            "label": row.label,
            "view": row.view,
            "split": row.split,                  # video의 split을 그대로 물려받음
            "has_groundtruth": row.has_groundtruth,
        })
    return frame_rows

all_frames = []
for _, row in df.iterrows():
    all_frames.extend(expand_to_frames(row))

frame_df = pd.DataFrame(all_frames)
print("총 프레임 수:", len(frame_df))
print(frame_df.split.value_counts())
frame_df.head()

총 프레임 수: 15799
split
train    13316
val       1396
test      1087
Name: count, dtype: int64


,patient_id,video_id,frame_path,frame_id,label,view,split,has_groundtruth
0,p1,v2,/content/CADICA/CADICA/selectedVideos/p1/v2/in...,p1_v2_00001,lesion,<bound method Series.view of patient_id ...,train,True
1,p1,v2,/content/CADICA/CADICA/selectedVideos/p1/v2/in...,p1_v2_00002,lesion,<bound method Series.view of patient_id ...,train,True
2,p1,v2,/content/CADICA/CADICA/selectedVideos/p1/v2/in...,p1_v2_00003,lesion,<bound method Series.view of patient_id ...,train,True
3,p1,v2,/content/CADICA/CADICA/selectedVideos/p1/v2/in...,p1_v2_00004,lesion,<bound method Series.view of patient_id ...,train,True
4,p1,v2,/content/CADICA/CADICA/selectedVideos/p1/v2/in...,p1_v2_00005,lesion,<bound method Series.view of patient_id ...,train,True


In [4]:
frame_df.to_csv("/content/mapped_frames.csv", index=False)

In [5]:
from torch.utils.data import Dataset
from PIL import Image

class CADICADataset(Dataset):
  def __init__(self, frame_df, transform=None):
    self.df = frame_df
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    image = Image.open(row['frame_path']).convert('RGB')
    label = 1 if row['label'] == 'lesion' else 0

    if self.transform:
      image = self.transform(image)

    return image, label


In [6]:
train_df = frame_df[frame_df['split'] == 'train']

train_dataset = CADICADataset(train_df, transform=None)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"데이터 로더가 성공적으로 생성되었습니다. 총 배치 수: {len(train_loader)}")

데이터 로더가 성공적으로 생성되었습니다. 총 배치 수: 417


In [7]:
from torchvision import transforms

def get_model_transforms(model_type='inception'):
  size = 299 if model_type=='inception' else 380
  return transforms.Compose([
      # transforms.RandomResizedCrop(
      #     size = size,
      #     scale = (0.8, 1.0) # 0.9,1.0 도 해보기
      # ),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(5),
      transforms.ColorJitter(
          brightness = 0.1,
          contrast=0.1,
          saturation=0.1,
          hue=0.05
      ),
      transforms.ToTensor(),
      transforms.Normalize(
          mean=[0.485,0.456,0.406],
          std = [0.229,0.224,0.225]
      )
  ])

In [8]:
import torch

train_transform = get_model_transforms(model_type='inception')

val_transform = transforms.Compose([
    transforms.Resize((299,299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_df = frame_df[frame_df['split'] == 'train']
val_df = frame_df[frame_df['split'] == 'val']

train_dataset = CADICADataset(train_df, transform=train_transform)
val_dataset = CADICADataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
    )
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
    )

In [9]:
import torch
import torch.nn as nn
from torchvision import models
# Fine_tunning
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.inception_v3(
    weights="DEFAULT",
    # aux_logits=True
)

# Freeze
for param in model.parameters():
    param.requires_grad = False

# for param in model.Mixed_7a.parameters():
#     param.requires_grad = True

# for param in model.Mixed_7b.parameters():
#     param.requires_grad = True

# 마지막 블록만 학습
for param in model.Mixed_7c.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

for param in model.fc.parameters():
    param.requires_grad = True
# aux_ftrs = model.AuxLogits.fc.in_features
# model.AuxLogits.fc = nn.Linear(aux_ftrs, 2)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 189MB/s]


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

criterion = nn.CrossEntropyLoss()

patience = 7
trigger_times = 0

best_f1 = 0.0

num_epochs = 50

for epoch in range(num_epochs):

    # Train
    model.train()
    running_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        if hasattr(outputs, "logits"):
            outputs = outputs.logits

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    # Validation
    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            probs = torch.softmax(outputs, dim=1)

            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            y_score.extend(probs[:, 1].cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)

    cm = confusion_matrix(y_true, y_pred)

    scheduler.step(f1)

    # Print

    print("=" * 50)
    print(f"Epoch [{epoch+1}/{num_epochs}]")

    print(f"Train Loss      : {running_loss/len(train_loader):.4f}")
    print(f"Train Accuracy  : {train_acc*100:.2f}%")

    print(f"Validation Accuracy : {accuracy*100:.2f}%")
    print(f"Precision           : {precision:.4f}")
    print(f"Recall              : {recall:.4f}")
    print(f"F1-score            : {f1:.4f}")
    print(f"ROC-AUC             : {auc:.4f}")

    print("\nConfusion Matrix")
    print(cm)

    # Save Best Model
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")
        print("Best Model Saved")
        trigger_times = 0

    else:
        trigger_times += 1
        print(f"\nEarlyStopping : {trigger_times}/{patience}")

        if trigger_times >= patience:
            print("Early Stopping")
            break

Epoch [1/50]
Train Loss      : 0.3134
Train Accuracy  : 87.12%
Validation Accuracy : 60.46%
Precision           : 0.7561
Recall              : 0.7045
F1-score            : 0.7294
ROC-AUC             : 0.4846

Confusion Matrix
[[100 240]
 [312 744]]
Best Model Saved
Epoch [2/50]
Train Loss      : 0.1572
Train Accuracy  : 94.16%
Validation Accuracy : 66.48%
Precision           : 0.7712
Recall              : 0.7917
F1-score            : 0.7813
ROC-AUC             : 0.5459

Confusion Matrix
[[ 92 248]
 [220 836]]
Best Model Saved
Epoch [3/50]
Train Loss      : 0.1002
Train Accuracy  : 96.66%
Validation Accuracy : 68.27%
Precision           : 0.7600
Recall              : 0.8485
F1-score            : 0.8018
ROC-AUC             : 0.5097

Confusion Matrix
[[ 57 283]
 [160 896]]
Best Model Saved
Epoch [4/50]
Train Loss      : 0.0721
Train Accuracy  : 97.65%
Validation Accuracy : 70.70%
Precision           : 0.7654
Recall              : 0.8835
F1-score            : 0.8202
ROC-AUC             : 0

In [20]:
print(y_true[:10])
print(y_pred[:10])
print(y_score[:10])

[np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
[np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
[np.float32(0.4293882), np.float32(0.010167526), np.float32(0.94095814), np.float32(0.114320114), np.float32(0.0047478443), np.float32(5.3521588e-05), np.float32(1.568248e-06), np.float32(0.008505161), np.float32(0.00024774089), np.float32(6.669623e-05)]


In [21]:
print(outputs[:5])

tensor([[-5.0986,  5.1267],
        [-0.5967,  0.2080],
        [-1.6892,  1.4176],
        [-4.5278,  4.2211],
        [-2.0741,  1.7603]], device='cuda:0')


In [22]:
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

label
lesion        9969
non-lesion    3347
Name: count, dtype: int64
label
lesion        1056
non-lesion     340
Name: count, dtype: int64


In [23]:
images, labels = next(iter(val_loader))

print(labels[:20])

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [24]:
print(torch.softmax(outputs[:10], dim=1))

tensor([[3.6241e-05, 9.9996e-01],
        [3.0903e-01, 6.9097e-01],
        [4.2826e-02, 9.5717e-01],
        [1.5860e-04, 9.9984e-01],
        [2.1157e-02, 9.7884e-01],
        [6.5543e-01, 3.4457e-01],
        [1.8568e-04, 9.9981e-01],
        [3.4121e-02, 9.6588e-01],
        [1.0276e-01, 8.9724e-01],
        [4.1612e-01, 5.8388e-01]], device='cuda:0')


In [25]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

train_labels = (train_df["label"] == "lesion").astype(int).values

class_count = train_df["label"].value_counts()

num_non = class_count["non-lesion"]
num_les = class_count["lesion"]

class_weights = {
    0: len(train_labels) / (2 * num_non),
    1: len(train_labels) / (2 * num_les)
}

sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [26]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.inception_v3(weights="DEFAULT")

# 전체 Freeze
for param in model.parameters():
    param.requires_grad = False

# 마지막 두 블록만 학습
for param in model.Mixed_7b.parameters():
    param.requires_grad = True

for param in model.Mixed_7c.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_ftrs, 2)
)

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

In [27]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)
weights = torch.tensor([1.0,3.0])
criterion = nn.CrossEntropyLoss()

patience = 7
trigger_times = 0
best_f1 = 0

num_epochs = 50

for epoch in range(num_epochs):

    # ---------------- Train ----------------
    model.train()

    running_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        if hasattr(outputs, "logits"):
            outputs = outputs.logits

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    # ---------------- Validation ----------------
    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch_idx, (images, labels) in enumerate(val_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            probs = torch.softmax(outputs, dim=1)

            # 첫 배치 확인
            if batch_idx == 0:
                print("\n===== Debug =====")
                print("Raw outputs")
                print(outputs[:5])

                print("\nSoftmax")
                print(probs[:5])

                print("\nPredicted")
                print(torch.argmax(probs[:5], dim=1))

                print("\nLabels")
                print(labels[:5])

            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            y_score.extend(probs[:, 1].cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_true,
        y_score
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    scheduler.step(f1)

    print("=" * 60)
    print(f"Epoch [{epoch+1}/{num_epochs}]")

    print(f"Train Loss      : {running_loss/len(train_loader):.4f}")
    print(f"Train Accuracy  : {train_acc*100:.2f}%")

    print(f"Validation Accuracy : {accuracy*100:.2f}%")
    print(f"Precision           : {precision:.4f}")
    print(f"Recall              : {recall:.4f}")
    print(f"F1-score            : {f1:.4f}")
    print(f"ROC-AUC             : {auc:.4f}")

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["non-lesion", "lesion"],
        digits=4,
        zero_division=0
    ))

    print("y_score min :", min(y_score))
    print("y_score max :", max(y_score))
    print("y_score mean:", sum(y_score)/len(y_score))

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")
        print("\nBest Model Saved")
        trigger_times = 0

    else:
        trigger_times += 1
        print(f"\nEarlyStopping : {trigger_times}/{patience}")

        if trigger_times >= patience:
            print("Early Stopping")
            break


===== Debug =====
Raw outputs
tensor([[ 0.1486, -0.4855],
        [ 0.4092, -0.5863],
        [-0.3983,  0.0051],
        [ 1.3029, -1.8579],
        [ 0.4812, -0.5601]], device='cuda:0')

Softmax
tensor([[0.6534, 0.3466],
        [0.7302, 0.2698],
        [0.4005, 0.5995],
        [0.9593, 0.0407],
        [0.7391, 0.2609]], device='cuda:0')

Predicted
tensor([0, 0, 1, 0, 0], device='cuda:0')

Labels
tensor([1, 1, 1, 1, 1], device='cuda:0')
Epoch [1/50]
Train Loss      : 0.6254
Train Accuracy  : 64.07%
Validation Accuracy : 70.85%
Precision           : 0.7805
Recall              : 0.8551
F1-score            : 0.8161
ROC-AUC             : 0.6763

Confusion Matrix
[[ 86 254]
 [153 903]]

Classification Report
              precision    recall  f1-score   support

  non-lesion     0.3598    0.2529    0.2971       340
      lesion     0.7805    0.8551    0.8161      1056

    accuracy                         0.7085      1396
   macro avg     0.5701    0.5540    0.5566      1396
weighted 

In [28]:
print(train_loader.sampler)

In [29]:
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

label
lesion        9969
non-lesion    3347
Name: count, dtype: int64
label
lesion        1056
non-lesion     340
Name: count, dtype: int64


In [30]:
count0 = 0
count1 = 0

for _, labels in train_loader:
    count0 += (labels == 0).sum().item()
    count1 += (labels == 1).sum().item()

print("non-lesion :", count0)
print("lesion :", count1)
# Weighted Random sampler 정상 적용 확인

non-lesion : 6796
lesion : 6520


In [31]:
len(train_dataset)

13316